# 1단계 · 시드 아이템 확정

**목표:** 분석의 출발점이 될 **시드(seed) 아이템**을 정합니다.

## 개념
- 국가전략기술 **카테고리**와 **지표**를 고르면, APOLLO가 해당 조합의 **TOP100** 아이템을 순위와 함께 돌려줍니다. (A4.1)
- 이 중 하나를 골라 이후 단계(네트워크 확장·상세 수집)의 시드로 씁니다.

## 왜 검색이 아니라 TOP100에서 고를까?
- APOLLO **검색(A4.2)**으로 얻은 `itemId`는 상세/네트워크 조회에서 오류(400)가 납니다.
- **TOP100(A4.1)** 의 `itemId`는 상세·네트워크 조회가 정상 동작합니다.
- 그래서 시드는 **반드시 TOP100 목록에서** 선택합니다.

## 지표 3종
- **기술집약도**: 기술적 성숙·특허 밀집도. 높을수록 진입장벽·응용 잠재력 큼.
- **수요부상도 / 공급부상도**: 시장 수요 / 연구·공급의 부상 속도.
- 💡 추세·네트워크가 풍부한 조합은 **기술집약도** 지표입니다.

In [ ]:
# --- APOLLO API 호출 헬퍼 (모든 노트북 공통) ---
import requests, urllib3
import pandas as pd
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://apollo.kisti.re.kr/service-test"   # 테스트 서버

def call_api(method, path, params=None, payload=None):
    """APOLLO 호출 후 {meta, data} 본문을 반환. (테스트 서버라 verify=False)"""
    r = requests.request(method, BASE_URL + path, params=params, json=payload,
                         headers={"Content-Type": "application/json"},
                         verify=False, timeout=120)
    body = r.json()
    if not (isinstance(body, dict) and "data" in body):
        raise RuntimeError(f"API 오류 (HTTP {r.status_code}): {body}")
    return body

print("준비 완료 · BASE_URL =", BASE_URL)

### TOP100 불러오기

In [ ]:
CATEGORY = "인공지능"
INDICATOR = "TECH_INTENSITY"      # 기술집약도

res = call_api("GET", "/open/api/v1/itemsntop100",
               params={"category": CATEGORY, "indicator": INDICATOR})
top = res["data"]["items"]

df = pd.DataFrame(top)[["rank", "itemId", "itemName", "score", "descriptionKor"]]
print(f"{CATEGORY} · {INDICATOR} TOP100:", len(df), "건")
df.head(10)

### 시드 확정 (여기서는 1위 아이템)

In [ ]:
SEED_RANK = 1
seed = top[SEED_RANK - 1]
SEED_ID = seed["itemId"]
SEED_NAME = seed["itemName"]
print("🌱 시드 확정:", SEED_NAME, "(itemId =", SEED_ID, ")")
print("기술집약도 점수:", seed.get("score"))
print()
print("설명:", (seed.get("descriptionKor") or "")[:300], "...")

**정리**
- 시드 = `Mathematical finance` (itemId `33589680`) 같은 TOP100 상위 아이템.
- 다음 단계에서 이 시드의 **연관 네트워크**를 API로 확장합니다.